# 01 — Exploratory Data Analysis

Overview of the REST-meta-MDD phenotypic data before any modeling.

**Goals**
- Class balance (MDD vs. HC)
- Site distribution and sample sizes
- Age and sex distributions per group
- Missing data audit
- Potential confounders (age, sex, site)

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import os

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
FIGURES_DIR = "../results/figures"
os.makedirs(FIGURES_DIR, exist_ok=True)

## 1. Load phenotypic data

In [ ]:
mdd  = pd.read_csv("../data/phenotypic/REST-meta-MDD-PhenotypicData_MDD.csv")
ctrl = pd.read_csv("../data/phenotypic/REST-meta-MDD-PhenotypicData_CONTROL.csv")

mdd["diagnosis"]  = 1
ctrl["diagnosis"] = 0

shared_cols = ["subject_id", "diagnosis", "Sex", "Age", "Education (years)"]
df = pd.concat([mdd[shared_cols], ctrl[shared_cols]], ignore_index=True)

# Extract site ID from subject_id (e.g. 'S15-2-0014' → 'S15')
df["site"] = df["subject_id"].str.extract(r"^(S\d+)-")

print(f"Total subjects : {len(df)}")
print(f"MDD            : {(df.diagnosis==1).sum()}")
print(f"HC             : {(df.diagnosis==0).sum()}")
df.head()

## 2. Class balance

In [ ]:
counts = df["diagnosis"].value_counts().rename({0: "HC", 1: "MDD"})

fig, ax = plt.subplots(figsize=(4, 3))
counts.plot(kind="bar", ax=ax, color=["#4C72B0", "#DD8452"], edgecolor="white", width=0.5)
ax.bar_label(ax.containers[0], fmt="%d", padding=3)
ax.set_title("Class distribution")
ax.set_ylabel("Subjects")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/eda_class_balance.png", dpi=150)
plt.show()

print(f"Imbalance ratio: {counts.max()/counts.min():.2f}")

## 3. Site distribution

Multi-site data introduces scanner/protocol variability — a major confound  
in neuroimaging studies. This plot shows sample size and MDD ratio per site.

In [ ]:
site_stats = (
    df.groupby("site")["diagnosis"]
    .agg(total="count", mdd="sum")
    .assign(hc=lambda d: d["total"] - d["mdd"],
            mdd_pct=lambda d: 100 * d["mdd"] / d["total"])
    .sort_values("total", ascending=False)
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: stacked bar — HC vs MDD per site
site_stats[["hc", "mdd"]].plot(
    kind="bar", stacked=True, ax=axes[0],
    color=["#4C72B0", "#DD8452"], edgecolor="white"
)
axes[0].set_title("Subjects per site")
axes[0].set_xlabel("Site")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=45)
axes[0].legend(["HC", "MDD"])

# Right: MDD percentage per site
axes[1].bar(site_stats.index, site_stats["mdd_pct"],
            color="#DD8452", edgecolor="white")
axes[1].axhline(50, color="gray", linestyle="--", linewidth=0.8)
axes[1].set_title("MDD % per site")
axes[1].set_xlabel("Site")
axes[1].set_ylabel("MDD (%)")
axes[1].set_ylim(0, 100)
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/eda_site_distribution.png", dpi=150)
plt.show()

print(site_stats.to_string())

## 4. Age and sex by group

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
label_map = {0: "HC", 1: "MDD"}
df["group"] = df["diagnosis"].map(label_map)

# Age distribution
sns.histplot(
    data=df, x="Age", hue="group", bins=25,
    kde=True, ax=axes[0],
    palette={"HC": "#4C72B0", "MDD": "#DD8452"}
)
axes[0].set_title("Age distribution by group")

# Sex distribution
sex_counts = (
    df.groupby(["group", "Sex"])
    .size()
    .reset_index(name="count")
)
sex_counts["Sex"] = sex_counts["Sex"].map({1: "Male", 2: "Female", 0: "Unknown"})

sns.barplot(
    data=sex_counts, x="group", y="count", hue="Sex",
    ax=axes[1],
    palette={"Male": "#4C72B0", "Female": "#DD8452", "Unknown": "#8C8C8C"}
)
axes[1].set_title("Sex distribution by group")
axes[1].set_xlabel("")

plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/eda_age_sex.png", dpi=150)
plt.show()

# Summary stats
df.groupby("group")["Age"].agg(["mean", "std", "min", "max"]).round(1)

## 5. Missing data audit

In [ ]:
missing = df[shared_cols].isna().sum().rename("missing")
missing_pct = (100 * missing / len(df)).rename("missing_%").round(1)

print(pd.concat([missing, missing_pct], axis=1).to_string())

## 6. Potential confounders

Age and sex differences between MDD and HC can confound classification.  
If significant, they should be regressed out or controlled in the model.

In [ ]:
from scipy import stats

mdd_age  = df.loc[df.diagnosis == 1, "Age"].dropna()
ctrl_age = df.loc[df.diagnosis == 0, "Age"].dropna()

t_stat, p_age = stats.ttest_ind(mdd_age, ctrl_age)
print(f"Age — MDD: {mdd_age.mean():.1f} ± {mdd_age.std():.1f}")
print(f"Age — HC : {ctrl_age.mean():.1f} ± {ctrl_age.std():.1f}")
print(f"t-test   : t={t_stat:.2f}, p={p_age:.4f}")
print()

# Sex: chi-square test
sex_table = pd.crosstab(df["diagnosis"], df["Sex"])
chi2, p_sex, _, _ = stats.chi2_contingency(sex_table)
print(f"Sex — chi2={chi2:.2f}, p={p_sex:.4f}")
print()
if p_age < 0.05 or p_sex < 0.05:
    print("⚠️  Significant demographic differences detected.")
    print("   Consider including Age and Sex as covariates in the model.")
else:
    print("✓  No significant demographic differences.")